In [ ]:
import cv2
import os
import ipywidgets as widgets
from IPython.display import display
from tkinter import Tk, filedialog
from tqdm import tqdm
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing as mp
from functools import partial

def save_frame_batch(frames_data, output_dir, video_basename, start_idx, jpeg_quality=85):
    # """Save a batch of frames to disk."""
    for i, frame in enumerate(frames_data):
        frame_filename = os.path.join(output_dir, f"{video_basename}_frame_{start_idx + i:04d}.jpg")
        cv2.imwrite(frame_filename, frame, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    return len(frames_data)

def extract_frames_optimized(video_path, output_dir, skip_start=0, skip_end=0, 
                            every_nth_frame=10, batch_size=100, num_workers=4, 
                            jpeg_quality=85):
    """
    Making use of parallel processing for optimal performance.
    
    Parameters:
    - video_path: Path to your video file
    - output_dir: Directory where your frames will be saved
    - skip_start: Seconds to skip at the start of the video
    - skip_end: Seconds to skip at the end of the video
    - every_nth_frame: Extract every nth frame (1 = all frames, 30 = 1fps for 30fps video for example)
    - batch_size: Number of frames to process in each batch. More means potentially slower processing times
    - num_workers: Number of parallel workers for saving frames. More can be faster but more CPU usage
    - jpeg_quality: JPEG compression quality (50-100, higher = better quality)
    """
    os.makedirs(output_dir, exist_ok=True)
    video_basename = os.path.splitext(os.path.basename(video_path))[0]
    
    # Open video with optimized settings
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 3)
    
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    # Get video properties 
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frame_count / fps
    
    print(f"Video info: {total_frame_count} frames, {fps} FPS, {duration:.2f} seconds")
    
    # Calculate frame range
    start_frame = int(skip_start * fps)
    end_frame = min(int((duration - skip_end) * fps), total_frame_count)
    
    if start_frame >= total_frame_count or end_frame <= start_frame:
        print("Error: Invalid skip_start or skip_end values.")
        cap.release()
        return

    # Create output subfolder for this video
    video_output_dir = os.path.join(output_dir, video_basename)
    os.makedirs(video_output_dir, exist_ok=True)
    
    # Calculate total frames to extract
    total_frames_to_extract = (end_frame - start_frame) // every_nth_frame
    output_fps = fps / every_nth_frame
    
    print(f"Extracting {total_frames_to_extract} frames (effective {output_fps:.1f} fps)")
    
    # Sequential reading with parallel saving
    frame_buffer = []
    frame_indices = []
    extracted_count = 0
    current_frame = 0
    output_frame_idx = 0
    
    # Skip to start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    current_frame = start_frame
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = []
        
        with tqdm(total=total_frames_to_extract, 
                 desc=f"Extracting frames", 
                 unit="frame") as progress:
            
            while current_frame < end_frame:
                ret, frame = cap.read()
                
                if not ret:
                    break
                
                # Backstops - Check if this frame can be extracted
                if (current_frame - start_frame) % every_nth_frame == 0:
                    frame_buffer.append(frame)
                    frame_indices.append(output_frame_idx)
                    output_frame_idx += 1
                    
                    # When buffer is full, submit batch for saving
                    if len(frame_buffer) >= batch_size:
                        batch_start_idx = frame_indices[0]
                        future = executor.submit(
                            save_frame_batch,
                            frame_buffer.copy(),
                            video_output_dir,
                            video_basename,
                            batch_start_idx,
                            jpeg_quality
                        )
                        futures.append(future)
                        
                        # Clear buffers
                        frame_buffer = []
                        frame_indices = []
                
                current_frame += 1
                
                # Update progress bar based on completed saves
                for future in as_completed(futures):
                    if future.done():
                        saved_count = future.result()
                        extracted_count += saved_count
                        progress.update(saved_count)
                        futures.remove(future)
            
            # Save remaining frames in buffer
            if frame_buffer:
                batch_start_idx = frame_indices[0]
                future = executor.submit(
                    save_frame_batch,
                    frame_buffer,
                    video_output_dir,
                    video_basename,
                    batch_start_idx,
                    jpeg_quality
                )
                futures.append(future)
            
            # Wait for all remaining saves to complete
            for future in as_completed(futures):
                saved_count = future.result()
                extracted_count += saved_count
                progress.update(saved_count)
    
    cap.release()
    print(f"✅ Extracted {extracted_count} frames to {video_output_dir}")
    return extracted_count

def create_extraction_ui():
    """UI for video frame extraction."""
    
    # File browsing functions
    def browse_video(button):
        root = Tk()
        root.withdraw()
        file_path = filedialog.askopenfilename(
            title="Select Video File",
            filetypes=[("Video files", "*.mp4 *.avi *.mov *.mkv *.wmv")]
        )
        if file_path:
            input_video.value = file_path
        root.destroy()

    def browse_output_folder(button):
        root = Tk()
        root.withdraw()
        folder_path = filedialog.askdirectory(title="Select Output Folder")
        if folder_path:
            output_folder.value = folder_path
        root.destroy()

    # Main widgets
    input_video = widgets.Text(
        description="Video file:", 
        placeholder="Path to your video file",
        style={'description_width': '120px'},
        layout={'width': '500px'}
    )
    input_button = widgets.Button(description="Browse", layout={'width': '80px'})
    input_button.on_click(browse_video)

    output_folder = widgets.Text(
        description="Output folder:", 
        placeholder="Where to save extracted frames",
        style={'description_width': '120px'},
        layout={'width': '500px'}
    )
    output_button = widgets.Button(description="Browse", layout={'width': '80px'})
    output_button.on_click(browse_output_folder)

    # Frame extraction slider 
    nth_frame = widgets.IntSlider(
        value=10,
        min=1,
        max=80,
        step=1,
        description='Frame interval:',
        style={'description_width': '120px'},
        layout={'width': '600px'}
    )
    
    # Label for nth_frame slider
    frame_info_label = widgets.HTML(value="<i>Extract every 10th frame</i>")
    
    def update_frame_info(change):
        n = change['new']
        if n == 1:
            frame_info_label.value = f"<i>Extract all frames (every frame)</i>"
        else:
            frame_info_label.value = f"<i>Extract every {n} frames (1/{n} of original framerate)</i>"
    
    nth_frame.observe(update_frame_info, 'value')

    # Tool description
    tool_description = widgets.HTML(
        value="""<h3>🎬 Video Frame Extractor</h3>
                <p>Extract frames from your video files, with parallel processing for fast performance.</p>
                <ul>
                    <li><b>Frame interval:</b> Controls sampling rate (10 = every 10th frame)</li>
                    <li><b>Output:</b> Frames saved as numbered JPEG files in subfolders</li>
                    <li><b>Performance:</b> Uses parallel processing for speed</li>
                </ul>"""
    )

    # Advanced options
    advanced_toggle = widgets.ToggleButton(
        value=False,
        description='Advanced Options',
        layout={'width': '150px'}
    )
    
    skip_start = widgets.IntSlider(
        value=0,
        min=0,
        max=500,
        step=5,
        description='Skip start (sec):',
        style={'description_width': '120px'},
        layout={'width': '400px'},
    )
    
    skip_end = widgets.IntSlider(
        value=0,
        min=0,
        max=500,
        step=5,
        description='Skip end (sec):',
        style={'description_width': '120px'},
        layout={'width': '400px'},
    )
    
    num_workers = widgets.IntSlider(
        value=min(4, mp.cpu_count()),
        min=1,
        max=mp.cpu_count(),
        step=1,
        description='CPU workers (power usage):',
        style={'description_width': '120px'},
        layout={'width': '400px'},
    )
    
    batch_size = widgets.IntSlider(
        value=100,
        min=10,
        max=500,
        step=10,
        description='Batch size:',
        style={'description_width': '120px'},
        layout={'width': '400px'},
    )
    
    jpeg_quality = widgets.IntSlider(
        value=85,
        min=50,
        max=100,
        step=5,
        description='JPEG quality:',
        style={'description_width': '120px'},
        layout={'width': '400px'},
    )
    
    advanced_box = widgets.VBox([
        widgets.HTML("<b>Timing:</b>"),
        widgets.HBox([skip_start, skip_end]),
        widgets.HTML("<b>Performance:</b>"),
        widgets.HBox([num_workers, batch_size]),
        widgets.HTML("<b>Quality:</b>"),
        jpeg_quality,
        widgets.HTML("<i>Higher quality = larger files</i>")
    ])
    advanced_box.layout.display = 'none'
    
    def toggle_advanced(change):
        if change['new']:
            advanced_box.layout.display = 'block'
            advanced_toggle.description = 'Hide Advanced'
        else:
            advanced_box.layout.display = 'none'
            advanced_toggle.description = 'Advanced Options'
    
    advanced_toggle.observe(toggle_advanced, 'value')
    
    # Run button and status
    status_output = widgets.Output()
    
    run_button = widgets.Button(
        description="Extract Frames",
        button_style='success',
        layout={'width': '150px'}
    )
    
    def run_extraction(button):
        with status_output:
            status_output.clear_output()
            
            video_path = input_video.value.strip()
            output_dir = output_folder.value.strip()
            
            if not video_path or not output_dir:
                print("❌ Please select both video file and output folder.")
                return
            
            if not os.path.exists(video_path):
                print(f"❌ Video file not found: {video_path}")
                return
                
            print(f"Starting frame extraction...")
            print(f"Video: {os.path.basename(video_path)}")
            print(f"Output: {output_dir}")
            print("-" * 40)
            
            try:
                frame_count = extract_frames_optimized(
                    video_path=video_path,
                    output_dir=output_dir,
                    skip_start=skip_start.value,
                    skip_end=skip_end.value,
                    every_nth_frame=nth_frame.value,
                    batch_size=batch_size.value,
                    num_workers=num_workers.value,
                    jpeg_quality=jpeg_quality.value
                )
                
                print(f"\n✅ Extraction completed! {frame_count} frames saved.")
                
            except Exception as e:
                print(f"❌ Error: {str(e)}")
                import traceback
                traceback.print_exc()
    
    run_button.on_click(run_extraction)
    
    return widgets.VBox([
        tool_description,
        widgets.HBox([input_video, input_button]),
        widgets.HBox([output_folder, output_button]),
        nth_frame,
        frame_info_label,
        advanced_toggle,
        advanced_box,
        run_button,
        status_output
    ])

def run_frame_extractor():
    """Run the frame extractor UI."""
    display(create_extraction_ui())

if __name__ == "__main__":
    run_frame_extractor()